# Data Exploration - Refugee Hosting Messages

This notebook provides an initial exploration of the ~20,000 messages from Singa's "Comme à la maison" platform.

## Objectives:
1. Load the data from Excel sheets
2. Understand the structure and content
3. Explore text characteristics
4. Identify patterns in motivations and descriptions
5. Prepare for emotion analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Load the Data

In [ ]:
# Data path
data_dir = Path('../data/raw')
excel_files = list(data_dir.glob('*.xlsx'))

if excel_files:
    data_path = excel_files[0]
    print(f"Loading data from: {data_path.name}")
else:
    print("No Excel file found in data/raw/")
    print("Please add the data file to the data/raw/ directory")

In [ ]:
# Load all three sheets
familles_1 = pd.read_excel(data_path, sheet_name='Familles 1')
familles_2 = pd.read_excel(data_path, sheet_name='Familles 2')
nouvelle_plateforme = pd.read_excel(data_path, sheet_name='Nouvelle Plateforme')

print(f"Familles 1: {len(familles_1):,} messages")
print(f"Familles 2: {len(familles_2):,} messages")
print(f"Nouvelle Plateforme: {len(nouvelle_plateforme):,} messages")
print(f"\nTotal: {len(familles_1) + len(familles_2) + len(nouvelle_plateforme):,} messages")

## 2. Explore Data Structure

In [ ]:
# Check columns in each sheet
print("Familles 1 columns:")
print(familles_1.columns.tolist())
print("\nFamilles 2 columns:")
print(familles_2.columns.tolist())
print("\nNouvelle Plateforme columns:")
print(nouvelle_plateforme.columns.tolist())

In [ ]:
# Display first few rows of Familles 2 (largest dataset)
familles_2.head(3)

## 3. Text Analysis: Motivations

In [ ]:
# Combine motivations from all sheets
all_motivations = pd.concat([
    familles_1['Motivations / Pourquoi CALM'],
    familles_2['Motivations / Pourquoi CALM'],
    nouvelle_plateforme['Motivations / Pourquoi CALM']
]).dropna()

print(f"Total non-empty motivations: {len(all_motivations):,}")
print(f"Average length: {all_motivations.str.len().mean():.0f} characters")
print(f"Median length: {all_motivations.str.len().median():.0f} characters")

In [ ]:
# Distribution of text lengths
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
all_motivations.str.len().hist(bins=50, edgecolor='black')
plt.xlabel('Message Length (characters)')
plt.ylabel('Frequency')
plt.title('Distribution of Motivation Text Length')
plt.axvline(all_motivations.str.len().median(), color='red', linestyle='--', label='Median')
plt.legend()

plt.subplot(1, 2, 2)
all_motivations.str.len().plot(kind='box')
plt.ylabel('Length (characters)')
plt.title('Box Plot of Text Lengths')

plt.tight_layout()
plt.show()

In [ ]:
# Sample messages
print("=== Sample Short Message ===")
short_messages = all_motivations[all_motivations.str.len() < 100]
if len(short_messages) > 0:
    print(short_messages.iloc[0])

print("\n=== Sample Medium Message ===")
medium_messages = all_motivations[(all_motivations.str.len() >= 100) & (all_motivations.str.len() < 300)]
if len(medium_messages) > 0:
    print(medium_messages.iloc[0])

print("\n=== Sample Long Message ===")
long_messages = all_motivations[all_motivations.str.len() >= 300]
if len(long_messages) > 0:
    print(long_messages.iloc[0][:500] + "...")

## 4. Geographic Distribution

In [ ]:
# Postal codes analysis
all_cps = pd.concat([
    familles_1['CP'],
    familles_2['CP'],
    nouvelle_plateforme['CP']
]).dropna()

# Convert to string and extract department (first 2 digits)
departments = all_cps.astype(str).str[:2]

print("Top 10 departments by number of messages:")
print(departments.value_counts().head(10))

In [ ]:
# Visualize top departments
plt.figure(figsize=(12, 6))
departments.value_counts().head(15).plot(kind='bar')
plt.xlabel('Department')
plt.ylabel('Number of Messages')
plt.title('Top 15 Departments by Message Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Hosting Characteristics

In [ ]:
# Hosting duration analysis (from Familles 2)
print("Hosting duration distribution:")
print(familles_2["Temps d'accueil"].value_counts().head(10))

In [ ]:
# Children acceptance
children_acceptance = pd.concat([
    familles_1['avec enfants ? '],
    familles_2['avec enfants ? '],
    nouvelle_plateforme['avec enfants ? ']
]).dropna()

print("\nChildren acceptance:")
print(children_acceptance.value_counts())

plt.figure(figsize=(8, 6))
children_acceptance.value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Acceptance of Hosting with Children')
plt.ylabel('')
plt.show()

## 6. Text Content Analysis

In [ ]:
# Common words in motivations (simple frequency)
from collections import Counter
import re

# Combine all text
all_text = ' '.join(all_motivations.astype(str).tolist())

# Simple tokenization (lowercase, remove punctuation)
words = re.findall(r'\b[a-zàâäéèêëïîôùûüÿœç]+\b', all_text.lower())

# Count words
word_freq = Counter(words)

# Remove common French stop words (simple list)
stop_words = {'le', 'la', 'les', 'un', 'une', 'des', 'de', 'du', 'et', 'est', 'à', 'en', 
              'que', 'qui', 'nous', 'vous', 'je', 'il', 'elle', 'dans', 'pour', 'sur', 
              'avec', 'par', 'ce', 'se', 'son', 'sa', 'ses', 'au', 'aux', 'mais', 'ou', 
              'car', 'ne', 'pas', 'd', 'l', 'j', 'ai', 'avons', 'ont', 'être', 'avoir',
              'mon', 'ma', 'mes', 'notre', 'nos', 'votre', 'vos', 'leur', 'leurs'}

# Filter stop words
filtered_words = {word: count for word, count in word_freq.items() 
                  if word not in stop_words and len(word) > 2}

# Top words
print("Top 30 most common words (excluding stop words):")
for word, count in sorted(filtered_words.items(), key=lambda x: x[1], reverse=True)[:30]:
    print(f"{word:20s}: {count:5d}")

In [ ]:
# Visualize top words
top_words = dict(sorted(filtered_words.items(), key=lambda x: x[1], reverse=True)[:20])

plt.figure(figsize=(12, 6))
plt.barh(list(top_words.keys()), list(top_words.values()))
plt.xlabel('Frequency')
plt.title('Top 20 Most Common Words in Motivations')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Emotional Keywords

In [ ]:
# Look for emotion-related keywords
emotion_keywords = {
    'solidarité': 0,
    'aide': 0,
    'aider': 0,
    'partage': 0,
    'partager': 0,
    'accueil': 0,
    'accueillir': 0,
    'famille': 0,
    'humanité': 0,
    'générosité': 0,
    'échange': 0,
    'rencontre': 0,
    'découverte': 0,
    'culture': 0,
    'choquée': 0,
    'choqué': 0,
    'triste': 0,
    'espoir': 0,
    'besoin': 0
}

for keyword in emotion_keywords.keys():
    emotion_keywords[keyword] = all_motivations.str.lower().str.contains(keyword, na=False).sum()

# Sort by frequency
sorted_keywords = dict(sorted(emotion_keywords.items(), key=lambda x: x[1], reverse=True))

print("Emotion-related keywords frequency:")
for word, count in sorted_keywords.items():
    print(f"{word:15s}: {count:5d} ({count/len(all_motivations)*100:.1f}%)")

In [ ]:
# Visualize emotional keywords
plt.figure(figsize=(12, 6))
plt.bar(sorted_keywords.keys(), sorted_keywords.values())
plt.xlabel('Keyword')
plt.ylabel('Frequency')
plt.title('Emotion-Related Keywords in Messages')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Next Steps

Based on this exploration, the next steps are:

1. **Prepare data for NLP pipeline**:
   - Clean and normalize text
   - Handle missing values
   - Create unified dataset

2. **Emotion Analysis**:
   - Test French emotion detection models
   - Extract emotional profiles
   - Map emotions over time (within each message)

3. **Text-to-Speech preparation**:
   - Select representative messages
   - Test TTS engines
   - Map emotions to prosody parameters

4. **Avatar design**:
   - Define facial expression parameters
   - Create emotion-to-expression mapping
   - Test animation synchronization

In [ ]:
# Save sample processed data
# Select a subset for initial testing
sample_df = familles_2[['CP', 'Motivations / Pourquoi CALM', 'Parlez nous de vous']].dropna(subset=['Motivations / Pourquoi CALM']).head(100)

# Save to processed directory
output_path = Path('../data/processed/sample_messages.csv')
sample_df.to_csv(output_path, index=False)
print(f"Sample data saved to: {output_path}")